# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Method choice and why

**Task:** Rank pages for review (binary label used as a proxy), judged by Precision@50.

**Models compared:**
- **Baseline (Week 4):** my hand-written score.
- **Logistic Regression:** simple and readable, so its coefficients show direction.
- **Random Forest:** handles non-linear signals and interactions with little tuning.

**Why these, not something fancier:** I only keep a more complex model if it clearly beats the simple one on the same split. Complexity alone isn't a result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATS = ["___feat1___", "___feat2___", "___feat3___", "___feat4___", "___feat5___"]
LABEL = "___label___"
GROUP = "___group_col___"   # client or site grouping column (anonymized); see the data dictionary

d = df.dropna(subset=FEATS + [LABEL]).reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr_idx, te_idx = next(gss.split(d, groups=d[GROUP]))
train, test = d.iloc[tr_idx], d.iloc[te_idx]

print("train:", len(train), "test:", len(test))
print("groups overlap:", len(set(train[GROUP]) & set(test[GROUP])))   # should print 0

## 2) Split design

I used a **grouped split** by ___ (no group appears in both train and test, checked above), so the model is tested on groups it hasn't seen.

Development uses month=2026-03 only. June 2026 stays sealed and is not touched in this notebook.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
def precision_at_k(y_true, score, k=50):
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())

# Baseline score from Week 4, restricted to the same test rows
base_test = test["score"] if "score" in test.columns else None
p_base = precision_at_k(test[LABEL], base_test, 50)
print("Baseline Precision@50:", round(p_base, 3), "| base rate:", round(test[LABEL].mean(), 3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

Xtr, ytr = train[FEATS], train[LABEL]
Xte, yte = test[FEATS], test[LABEL]

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1),
}

rows = [{"Method": "Baseline rule (Week 4)", "Precision@50": round(p_base, 3), "AUC": None}]
fitted, scores = {}, {}
for name, m in models.items():
    m.fit(Xtr, ytr)
    s = m.predict_proba(Xte)[:, 1]
    fitted[name], scores[name] = m, s
    rows.append({"Method": name,
                 "Precision@50": round(precision_at_k(yte, s, 50), 3),
                 "AUC": round(roc_auc_score(yte, s), 3)})

comparison = pd.DataFrame(rows)
comparison

In [ ]:
from sklearn.inspection import permutation_importance

best = max(scores, key=lambda n: precision_at_k(yte, scores[n], 50))
print("Best by Precision@50:", best)

r = permutation_importance(fitted[best], Xte, yte, n_repeats=5, random_state=0, scoring="roc_auc")
imp = pd.Series(r.importances_mean, index=FEATS).sort_values(ascending=False).round(4)
imp

In [ ]:
test = test.copy()
test["model_score"] = scores[best]
top = test.sort_values("model_score", ascending=False).head(50)

false_pos = top[top[LABEL] == 0]
print("False positives in top 50:", len(false_pos))
false_pos[[GROUP] + FEATS + ["model_score"]].head(10)

## 3) Model vs baseline (same split, same metric)

(The table above.) Precision@50: baseline ___, Logistic Regression ___, Random Forest ___.

**My read:** ___ (for example, "the Random Forest beats the baseline by ___, but Logistic Regression gets most of the way there, so the extra complexity buys ___.")

## 4) Errors and interpretation

- The most important features were ___ and ___ (permutation importance, measured on the test split).
- The false positives in the top 50 tend to ___ (for example, have a high ___ but a recent ___).
- This suggests the model misses ___, which a future feature could capture.

**Careful words:** These results are observed on a sample and are directional. Feature importance describes the model, not causes in search.

## 5) Self-check

- [x] Compared to the Week 4 baseline on the same split and metric
- [x] Used a grouped (or time-aware) split, with the overlap check printed
- [x] Explained why these methods
- [x] Reported Precision@50 and AUC
- [x] Interpreted features and errors
- [x] Didn't reward complexity alone

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.